# Term Deposit Marketing - Modeling

### Data Load

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

In [2]:
file_path = r"../data/term-deposit-marketing-2020.csv"
df = pd.read_csv(file_path)
# df.info()
# df.describe()
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,no


### Prepare Features

In [ ]:
# Target
y = (df["y"] == "yes").astype(int)
X = df.drop(columns=["y", "duration", "jobs"])

numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object", "str"]).columns

In [5]:
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
print("Shape: ", X_encoded.shape)
X_encoded.head()

Shape:  (40000, 35)


,age,balance,day,campaign,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,...,month_aug,month_dec,month_feb,month_jan,month_jul,month_jun,month_mar,month_may,month_nov,month_oct
0,58,2143,5,1,False,False,False,True,False,False,...,False,False,False,False,False,False,False,True,False,False
1,44,29,5,1,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,33,2,5,1,False,True,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,47,1506,5,1,True,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
4,33,1,5,1,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False


In [7]:
# Correlation with Target
corr_with_target = X_encoded.corrwith(y).sort_values(ascending=False)
corr_with_target

month_mar              0.128125
month_oct              0.093298
marital_single         0.051721
education_tertiary     0.046763
job_student            0.037384
month_feb              0.036816
balance                0.030232
job_retired            0.024343
job_management         0.020291
job_unemployed         0.009463
job_self-employed      0.005029
month_dec              0.000315
job_technician         0.000234
job_unknown           -0.000018
contact_telephone     -0.000842
day                   -0.006420
default_yes           -0.006559
job_entrepreneur      -0.007191
education_unknown     -0.007464
month_nov             -0.013654
month_jun             -0.014557
job_services          -0.014645
job_housemaid         -0.015248
education_secondary   -0.019683
age                   -0.020273
month_jul             -0.020528
month_aug             -0.025661
month_jan             -0.026922
loan_yes              -0.031029
job_blue-collar       -0.032859
month_may             -0.038479
campaign

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,10))
sns.heatmap(X_encoded.corr(), center=0)
plt.title("Correlation Matrix (Encoded Features)")
plt.show()


### Feature Engineering

In [ ]:
# X_encoded["high_balance_flag"] = (df["balance"] > df["balance"].median()).astype(int)
# X_encoded["is_student_or_retired"] = df["job"].isin(["student", "retired"]).astype(int)
# X_encoded["low_campaign_flag"] = (df["campaign"] <= 2).astype(int)
# X_encoded.drop(["job_unknown", "education_unknown", "contact_unknown"
# 
# 
# "job_unemployed"         0.009463
"job_self"-employed      0.005029
"month_dec"              0.000315
"job_technician"         0.000234
"job_unknown"           -0.000018
"contact_telephone"], axis=1, inplace=True)
# X_encoded.head()

### Baseline Logistic Model

In [ ]:
model = LogisticRegression(max_iter=20000)

pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", model)
])

#### 5-Fold Stratified CV

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score),
    "recall": make_scorer(recall_score),
    "f1": make_scorer(f1_score),
}

cv_results = cross_validate(
    X_encoded,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

results = {
    metric: np.mean(cv_results[f"test_{metric}"])
    for metric in scoring.keys()
}

results

#### Balanced Data

In [ ]:
model_balanced = LogisticRegression(
    max_iter=10000,
    class_weight="balanced",
    random_state=42
)

pipeline_balanced = Pipeline([
    ("preprocess", preprocess),
    ("model", model_balanced)
])

In [ ]:
cv_results_bal = cross_validate(
    pipeline_balanced,
    X,
    y,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

results_bal = {
    metric: np.mean(cv_results_bal[f"test_{metric}"])
    for metric in scoring.keys()
}

results_bal

#### Probability Ranking Evaluation

In [ ]:
pipeline_balanced.fit(X, y)
probs = pipeline_balanced.predict_proba(X)[:, 1]

df_eval = df.copy()
df_eval["pred_prob"] = probs

In [ ]:
# Create Deciles
df_eval["decile"] = pd.qcut(df_eval["pred_prob"], 10, labels=False)
lift_table = df_eval.groupby("decile")["y"].apply(lambda x: (x=="yes").mean()).sort_index(ascending=False)
lift_table